# NAM 实验 - GitHub 自动同步版本

**仓库**: https://github.com/yaoyuanArtemis/HKU-NAM-7404

**优势**: 本地修改代码后，只需 `git push`，Colab 自动同步！

## 🔄 工作流程

```
本地修改代码 → git push → GitHub → Colab git pull → 自动更新✅
```

## 1️⃣ 克隆项目（首次运行）

In [ ]:
import os

# GitHub 配置
GITHUB_REPO = 'https://github.com/yaoyuanArtemis/HKU-NAM-7404.git'
REPO_DIR = '/content/HKU-NAM-7404'
PROJECT_DIR = '/content/HKU-NAM-7404/neural_additive_models'

# 检查是否已克隆
if os.path.exists(REPO_DIR):
    print(f"✓ 项目已存在: {REPO_DIR}")
    print("跳过克隆，执行下一个单元格更新代码")
else:
    print("正在从 GitHub 克隆项目...")
    !git clone {GITHUB_REPO} {REPO_DIR}
    print(f"✓ 项目已克隆到: {REPO_DIR}")

# 切换到项目目录
os.chdir(PROJECT_DIR)
print(f"\n当前目录: {os.getcwd()}")
print("\n项目文件:")
!ls -lh *.py | head -10

## 2️⃣ 更新代码（每次运行前执行）⭐

**在本地修改代码并 `git push` 后，运行这个单元格同步到 Colab**

In [ ]:
import os
os.chdir('/content/HKU-NAM-7404/neural_additive_models')

print("正在从 GitHub 拉取最新代码...\n")

# 从 GitHub 拉取最新代码（master 分支）
!git pull origin master

print("\n✓ 代码已更新到最新版本！")
print("\n最近的提交:")
!git log -3 --oneline

## 3️⃣ 安装依赖

In [ ]:
!pip install -q xgboost scikit-learn interpret pandas numpy matplotlib
print("✓ 依赖安装完成")

## 4️⃣ 运行实验

### 方案 A: 单数据集实验（快速测试）

In [ ]:
# 准备测试数据
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.to_csv('breast_cancer.csv', index=False)

print(f"✓ 数据集准备完成: {df.shape}")
print(f"  样本数: {len(df)}")
print(f"  特征数: {len(data.feature_names)}")

In [ ]:
# 运行单数据集实验
!python run_experiment.py \
    --data_path breast_cancer.csv \
    --target_column target \
    --task classification

### 方案 B: 批量运行所有 NAM 数据集（完整实验）

In [ ]:
# 批量运行所有数据集
# 注意：这会需要 10-30 分钟
!python run_all_datasets.py

## 5️⃣ 查看结果

In [ ]:
import pandas as pd
import glob

# 查找结果文件
result_files = glob.glob('comparison_results/*_comparison.csv')

if result_files:
    latest_result = sorted(result_files)[-1]
    print(f"读取结果: {latest_result}\n")
    
    results_df = pd.read_csv(latest_result)
    print("="*70)
    print("模型对比结果")
    print("="*70)
    print(results_df.to_string(index=False))
else:
    print("未找到结果文件，请先运行实验")

## 6️⃣ 可视化结果

In [ ]:
import matplotlib.pyplot as plt

if result_files:
    results_df = pd.read_csv(latest_result)
    
    # 性能对比
    test_metric_cols = [col for col in results_df.columns if 'Test' in col and ('AUROC' in col or 'RMSE' in col)]
    
    if test_metric_cols:
        metric_col = test_metric_cols[0]
        
        plt.figure(figsize=(10, 6))
        plt.barh(results_df['Model'], results_df[metric_col])
        plt.xlabel(metric_col)
        plt.title('模型性能对比')
        plt.tight_layout()
        plt.show()
        
        # 训练时间对比
        if 'Train Time (s)' in results_df.columns:
            plt.figure(figsize=(10, 6))
            plt.barh(results_df['Model'], results_df['Train Time (s)'])
            plt.xlabel('训练时间 (秒)')
            plt.title('模型训练时间对比')
            plt.tight_layout()
            plt.show()

## 7️⃣ 下载结果

In [ ]:
from google.colab import files

# 打包所有结果
!zip -r results.zip comparison_results/ all_results/ 2>/dev/null || echo "部分目录不存在"

# 下载
files.download('results.zip')

print("✓ 结果已打包并开始下载")

---

## 💡 本地如何更新代码

### 在本地终端执行：

```bash
cd /Users/sh01679ml/Desktop/workding-code/google-research/neural_additive_models

# 查看修改
git status

# 提交修改
git add .
git commit -m "更新代码"
git push origin master
```

### 在 Colab 中：

重新运行 **「2️⃣ 更新代码」** 单元格即可！

---

## 🚀 一键更新并运行

In [ ]:
# 一键：更新代码 + 准备数据 + 运行实验
import os
os.chdir('/content/HKU-NAM-7404/neural_additive_models')

print("1️⃣ 更新代码...")
!git pull origin master

print("\n2️⃣ 准备数据...")
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.to_csv('test_data.csv', index=False)

print("\n3️⃣ 运行实验...")
!python run_experiment.py \
    --data_path test_data.csv \
    --target_column target \
    --task classification

print("\n✅ 完成！")

## ⚠️ 常见问题

### Q: git pull 提示冲突？

运行这个强制更新：

In [ ]:
# 强制覆盖为远程版本
!git fetch origin
!git reset --hard origin/master
print("✓ 已强制更新到远程版本")

### Q: 查看当前代码版本

In [ ]:
!git log -1 --format="%H - %s (%ar)"